# 심화 미션: 전기차 배터리 셀 최종검사
- 상황: 놓친 불합격 하나가 리콜로 이어지는 현장이다
- 목표: 오늘 배운 순서를 다른 데이터로 혼자 한 바퀴 돌린다

### 용어 풀이 - 배터리 검사에서 쓰는 말

| 말 | 뜻 |
|---|---|
| 개방 전압 | 아무것도 연결하지 않은 상태에서 잰 전압 (V) |
| 용량 | 이 셀이 담을 수 있는 전기의 양 (mAh). 클수록 오래 간다 |
| 내부 저항 | 전기가 흐를 때 셀 안에서 생기는 저항 (mΩ). 높을수록 열이 나고 성능이 떨어진다 |
| 두께 부풀음 | 셀이 규격보다 두꺼워지는 것. 안에서 가스가 생겼다는 신호일 수 있다 |
| 완충 시간 | 다 채우는 데 걸린 시간 (분). 오래 걸릴수록 어딘가 문제가 있을 수 있다 |
| 합격 / 불합격 | 검사실에서 붙이는 최종 판정 |

## 공통 준비

In [1]:
import pandas as pd

df = pd.read_csv("../../data/day04_battery.csv")

## Q1. 파일 열고 크기 확인하기

In [2]:
불합격수 = (df["result"] == "불합격").sum()

print(f"행 {df.shape[0]} / 열 {df.shape[1]}")
print(f"불합격 {불합격수}건 ({round(불합격수 / len(df) * 100, 1)}%)")

행 2847 / 열 14
불합격 133건 (4.7%)


## Q2. 빈칸 찾아 채우기

In [3]:
검사값열 = ["open_voltage_V", "capacity_mAh", "internal_resistance_mOhm", "thickness_mm",
          "weight_g", "charge_time_min", "chamber_temp_C", "chamber_humidity_pct"]

빈칸수 = df[검사값열].isna().sum()

print("채우기 전")
for 열, 개수 in 빈칸수[빈칸수 > 0].items():   # 빈칸이 있는 열만 골라 돈다
    print(f"  {열}: {개수}개")

for 열 in 검사값열:
    df[열] = df[열].fillna(df[열].median())   # 그 열의 중앙값으로 채운다

print(f"채운 뒤 남은 빈칸: {df[검사값열].isna().sum().sum()}개")

채우기 전
  charge_time_min: 38개
  chamber_temp_C: 57개
  chamber_humidity_pct: 113개
채운 뒤 남은 빈칸: 0개


## Q3. 입력과 정답 가르기

In [4]:
df["불합격여부"] = (df["result"] == "불합격").astype(int)

X = df[검사값열]        # Q2에서 만들어둔 여덟 개 목록을 그대로 쓴다
y = df["불합격여부"]

print("입력 크기:", X.shape)
print(f"정답 분포: 0이 {(y == 0).sum()}건, 1이 {(y == 1).sum()}건")

입력 크기: (2847, 8)
정답 분포: 0이 2714건, 1이 133건


## Q4. 학습용과 시험용으로 나누기

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)   # stratify=y 가 비율을 맞춘다

for 이름, 정답 in [("학습용", y_train), ("시험용", y_test)]:
    print(f"{이름} {len(정답)}건 (불합격 {정답.sum()}건, {round(정답.mean() * 100, 2)}%)")

학습용 2277건 (불합격 106건, 4.66%)
시험용 570건 (불합격 27건, 4.74%)


## Q5. 기준 모델 세우기

In [6]:
맞힌수 = (y_test == 0).sum()     # 전부 합격이라 답하면 실제 합격만 맞는다

print(f"기준 모델 정확도: {round(맞힌수 / len(y_test) * 100, 2)}%")
print("불합격이라 지목한 건수: 0건")

기준 모델 정확도: 95.26%
불합격이라 지목한 건수: 0건


## Q6. 손대지 않은 모델로 한 번

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

모델 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
모델.fit(X_train, y_train)
예측 = 모델.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, 예측).ravel()   # 네 칸을 순서대로 받는다

print(f"정확도 {round((예측 == y_test).mean() * 100, 2)}%")
print(f"잡은 불합격 {tp} / 놓친 불합격 {fn} / 헛경보 {fp}")
print(f"재현율 {recall_score(y_test, 예측):.3f}  정밀도 {precision_score(y_test, 예측):.3f}  F1 {f1_score(y_test, 예측):.3f}")

정확도 97.02%
잡은 불합격 12 / 놓친 불합격 15 / 헛경보 2
재현율 0.444  정밀도 0.857  F1 0.585


## Q7. 드문 쪽에 무게를 주고 다시

In [8]:
모델가중 = make_pipeline(StandardScaler(),
                     LogisticRegression(max_iter=1000, class_weight="balanced"))
모델가중.fit(X_train, y_train)
예측가중 = 모델가중.predict(X_test)

def 재보기(이름, 어떤예측):                      # 같은 항목을 두 번 재야 하니 함수로 묶는다
    tn, fp, fn, tp = confusion_matrix(y_test, 어떤예측).ravel()
    return {"처리": 이름,
            "정확도": f"{round((어떤예측 == y_test).mean() * 100, 2)}%",
            "잡은 불합격": tp, "놓친 불합격": fn, "헛경보": fp,
            "재현율": round(recall_score(y_test, 어떤예측), 3),
            "정밀도": round(precision_score(y_test, 어떤예측), 3),
            "F1": round(f1_score(y_test, 어떤예측), 3)}

표 = pd.DataFrame([재보기("손 안 댐", 예측), 재보기("가중치", 예측가중)])
print(표.to_string(index=False))

   처리    정확도  잡은 불합격  놓친 불합격  헛경보   재현율   정밀도    F1
손 안 댐 97.02%      12      15    2 0.444 0.857 0.585
  가중치 86.32%      25       2   76 0.926 0.248 0.391


### 무엇이 오르고 무엇이 내렸나

- 오른 것 : 재현율 0.444 → 0.926. 놓친 불합격이 15건에서 2건으로 줄었다
- 내린 것 : 정확도 97.02% → 86.32%, 정밀도 0.857 → 0.248. 헛경보가 2건에서 76건으로 늘었다
- F1도 내려갔다 (0.585 → 0.391). 그래도 한 과장이 원한 것은 **가중치 쪽**이다 —
  "놓치는 걸 최대한 줄이는 쪽으로, 좀 헛짚어도 괜찮다"고 했다

> 헛경보 74건이 늘어난 대가로 놓친 것을 13건 줄였다. 리콜 한 건과 재검 여섯 번을 맞바꾼 셈이고,
> 이 교환이 남는 장사인지는 현장이 판단한다.

## Q8. 다이얼 세 번 돌리기

In [9]:
from sklearn.tree import DecisionTreeClassifier

줄들 = []
for 깊이 in [3, 5, None]:                       # None 이 "제한 없음"이다
    나무 = DecisionTreeClassifier(max_depth=깊이, class_weight="balanced", random_state=42)
    나무.fit(X_train, y_train)
    예측나무 = 나무.predict(X_test)

    tn, fp, fn, tp = confusion_matrix(y_test, 예측나무).ravel()
    줄들.append({"깊이": "제한 없음" if 깊이 is None else 깊이,
               "정확도": f"{round((예측나무 == y_test).mean() * 100, 2)}%",
               "잡은 불합격": tp, "헛경보": fp,
               "재현율": round(recall_score(y_test, 예측나무), 3),
               "F1": round(f1_score(y_test, 예측나무), 3)})

print(pd.DataFrame(줄들).to_string(index=False))

   깊이    정확도  잡은 불합격  헛경보   재현율    F1
    3 88.42%      24   63 0.889 0.421
    5 85.79%      19   73 0.704 0.319
제한 없음 92.98%      11   24 0.407 0.355


### 다이얼은 매끈하게 움직이지 않는다

깊이 5는 깊이 3보다 **잡은 것도 적고(19 대 24) 헛경보는 더 많다(73 대 63).** 어느 쪽으로도 나은 게 없다.
"깊게 할수록 이렇게 된다"는 법칙이 없다는 뜻이고, 그래서 다음 문항에서 스물네 조합을 다 돌려본다.

## Q9. 자동 탐색으로 설정 찾기

In [10]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

겹나누기 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
후보 = {"max_depth": [2, 3, 4, 5, 10, None], "min_samples_leaf": [1, 5, 10, 20]}

탐색 = GridSearchCV(DecisionTreeClassifier(class_weight="balanced", random_state=42),
                  후보, scoring="f1", cv=겹나누기)
탐색.fit(X_train, y_train)                     # 학습용만 넣는다

print(f"1등 설정: 깊이 {탐색.best_params_['max_depth']}, 끝자리 최소 {탐색.best_params_['min_samples_leaf']}")
print(f"탐색 점수(F1): {round(탐색.best_score_, 3)}")

예측탐색 = 탐색.best_estimator_.predict(X_test)   # 1등 설정으로 이미 학습된 나무
tn, fp, fn, tp = confusion_matrix(y_test, 예측탐색).ravel()

print()
print("시험용 채점")
print(f"  정확도 {round((예측탐색 == y_test).mean() * 100, 2)}%  잡은 불합격 {tp}  헛경보 {fp}")
print(f"  재현율 {recall_score(y_test, 예측탐색):.3f}  정밀도 {precision_score(y_test, 예측탐색):.3f}  F1 {f1_score(y_test, 예측탐색):.3f}")

1등 설정: 깊이 10, 끝자리 최소 1
탐색 점수(F1): 0.38

시험용 채점
  정확도 91.4%  잡은 불합격 15  헛경보 37
  재현율 0.556  정밀도 0.288  F1 0.380


### ⛔ 시험용 점수를 보고 설정을 고르면 안 된다

시험용 F1을 보면 탐색 1등(깊이 10)은 0.380인데 Q8의 깊이 3은 0.421이다. 손으로 돌린 게 더 좋아 보인다.
그렇다고 깊이 3으로 바꾸면 **시험지를 열어보고 답을 고친 것**이 되고, 그 뒤로는 그 시험지로 점수를 잴 수 없다.

바르게 적으면 이렇다 — "학습용 안에서 F1 기준으로 고른 설정은 깊이 10·끝자리 1이고,
봉인해둔 시험용에서 F1 0.380이 나왔다." 깊이 3이 좋아 보인다는 건 참고로만 적어두고,
정말 궁금하면 다음에 데이터를 새로 받았을 때 확인한다.

## Q10. 교차검증으로 마무리

In [11]:
from sklearn.model_selection import cross_val_score

모델들 = {
    "손 안 댐": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "가중치": make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=1000, class_weight="balanced")),
    "나무(탐색 1등)": DecisionTreeClassifier(max_depth=10, min_samples_leaf=1,
                                       class_weight="balanced", random_state=42),
}

줄들 = []
for 이름, 어떤모델 in 모델들.items():
    재현율점수 = cross_val_score(어떤모델, X_train, y_train, cv=겹나누기, scoring="recall")
    F1점수 = cross_val_score(어떤모델, X_train, y_train, cv=겹나누기, scoring="f1")

    # float( ) 로 감싸는 이유 - 안 감싸면 목록 안에서 np.float64(0.859) 처럼 이름표가 붙는다
    줄들.append({"모델": 이름,
               "재현율 평균": round(float(재현율점수.mean()), 3),
               "재현율 흔들림": round(float(재현율점수.std()), 3),
               "F1 평균": round(float(F1점수.mean()), 3),
               "F1 흔들림": round(float(F1점수.std()), 3)})

print(pd.DataFrame(줄들).to_string(index=False))

       모델  재현율 평균  재현율 흔들림  F1 평균  F1 흔들림
    손 안 댐   0.406    0.064  0.517   0.085
      가중치   0.859    0.064  0.385   0.029
나무(탐색 1등)   0.500    0.067  0.380   0.050


In [12]:
# 참고 - 평균 뒤에 가려진 겹마다의 재현율도 꺼내본다
for 이름, 어떤모델 in 모델들.items():
    점수들 = cross_val_score(어떤모델, X_train, y_train, cv=겹나누기, scoring="recall")
    print(f"{이름:12s} {[round(float(v), 3) for v in 점수들]}")

손 안 댐        [0.381, 0.409, 0.524, 0.381, 0.333]
가중치          [0.857, 0.773, 0.81, 0.905, 0.952]
나무(탐색 1등)    [0.571, 0.545, 0.476, 0.524, 0.381]


### 자를 바꾸면 1등이 바뀐다

F1으로 보면 손 안 댄 쪽이 0.517로 제일 높다. 재현율로 보면 가중치 쪽이 0.859로 1등이다.
**어느 자로 볼지 먼저 정하고** 표를 읽어야 한다. 한 과장의 자는 재현율이다.

## 마무리 - 한 과장에게 드릴 말

- 권하는 것 : [가중치를 준 로지스틱 회귀]
- 왜 : [놓치는 걸 줄이는 게 급하다고 하셨고, 다섯 번 재서 평균 재현율 0.859로 가장 높았다]
- 내주는 것 : [헛경보가 76건 생긴다. 멀쩡한 셀을 570건 중 76건 다시 보게 된다]

## 오늘 막힌 곳

- 막힌 문항 : [Q3]
- 어디서 : [입력에 넣을 열을 고를 때 result와 불합격여부를 빼는 것을 놓쳐서 열이 9개가 됐다]

## 오늘 이 미션에서 확인한 것

- 소재가 바뀌어도 **순서는 그대로**다 — 빈칸 채우기 → 입력·정답 가르기 → 층화로 나누기 → 기준 모델 → 처리 → 다이얼 → 교차검증
- **드문 쪽이 드물수록 눈 감은 사람의 점수가 올라간다** — 기준 모델이 95.26점이었다
- **드문 쪽을 무겁게 세면 놓친 것이 줄고 헛경보가 는다.** 이 데이터에서도 예외 없었다
- **F1이 내려가도 그쪽을 고를 수 있다** — 현장이 무엇을 아파하느냐가 자를 정한다
- **다이얼은 매끈하게 움직이지 않는다** — 깊이 5가 깊이 3보다 모든 면에서 나빴다
- **자동 탐색이 시험용에서 늘 이기는 것도 아니다.** 그렇다고 시험용을 보고 고르면 안 된다
- 점수는 **평균과 흔들림을 함께** 적는다